# 01 · Explore the raw data

Before cleaning anything, this notebook checks what is actually in the files from `data/raw/`:

1. Row counts and column types
2. Missing values (nulls)
3. Hidden missing values: `-1` codes that mean "Unknown"

Run `python src/download.py` first to fetch the files.

In [1]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("..") / "data" / "raw"
pd.set_option("display.width", 120)

## Load the files

The three PriceCatcher months are combined into one table so every check covers all of them.

In [2]:
cpi = pd.read_parquet(RAW_DIR / "cpi_2d_state.parquet")
lookup_item = pd.read_parquet(RAW_DIR / "lookup_item.parquet")
lookup_premise = pd.read_parquet(RAW_DIR / "lookup_premise.parquet")
mcoicop = pd.read_parquet(RAW_DIR / "mcoicop.parquet")

price_files = sorted(RAW_DIR.glob("pricecatcher_*.parquet"))
prices = pd.concat([pd.read_parquet(f) for f in price_files], ignore_index=True)

tables = {
    "prices": prices,
    "cpi": cpi,
    "lookup_item": lookup_item,
    "lookup_premise": lookup_premise,
    "mcoicop": mcoicop,
}
print("PriceCatcher files:", [f.name for f in price_files])

PriceCatcher files: ['pricecatcher_2025-09.parquet', 'pricecatcher_2025-10.parquet', 'pricecatcher_2025-11.parquet', 'pricecatcher_2025-12.parquet', 'pricecatcher_2026-01.parquet', 'pricecatcher_2026-02.parquet', 'pricecatcher_2026-03.parquet', 'pricecatcher_2026-04.parquet', 'pricecatcher_2026-05.parquet', 'pricecatcher_2026-06.parquet', 'pricecatcher_2026-07.parquet', 'pricecatcher_2026-08.parquet', 'pricecatcher_2026-09.parquet']


## 1. Row counts and column types

In [3]:
pd.DataFrame(
    {name: {"rows": len(df), "columns": df.shape[1]} for name, df in tables.items()}
).T

,rows,columns
prices,22682970,4
cpi,45024,4
lookup_item,797,5
lookup_premise,3932,6
mcoicop,343,7


In [4]:
for name, df in tables.items():
    print(f"--- {name}")
    print(df.dtypes.to_string())
    print()

--- prices
date             object
premise_code      int64
item_code         int64
price           float64

--- cpi
state                  str
date        datetime64[ns]
division               str
index              float64

--- lookup_item
item_code        int64
item               str
unit               str
item_group         str
item_category      str

--- lookup_premise
premise_code    int64
premise           str
address           str
premise_type      str
state             str
district          str

--- mcoicop
digits      int64
division      str
group         str
class         str
subclass      str
desc_en       str
desc_bm       str



## 2. Missing values (nulls)

For each table: how many nulls each column has, and what percentage of rows that is.

In [5]:
def null_summary(df):
    """Null count and percentage for every column."""
    nulls = df.isna().sum()
    return pd.DataFrame({"nulls": nulls, "null_pct": (nulls / len(df) * 100).round(2)})


for name, df in tables.items():
    print(f"--- {name}")
    print(null_summary(df).to_string())
    print()

--- prices


              nulls  null_pct
date              0       0.0
premise_code      0       0.0
item_code         0       0.0
price             0       0.0

--- cpi
          nulls  null_pct
state         0       0.0
date          0       0.0
division      0       0.0
index         0       0.0

--- lookup_item
               nulls  null_pct
item_code          0      0.00
item               1      0.13
unit               1      0.13
item_group         1      0.13
item_category      1      0.13

--- lookup_premise
              nulls  null_pct
premise_code      0      0.00
premise           0      0.00
address           0      0.00
premise_type      1      0.03
state             1      0.03
district          1      0.03

--- mcoicop
          nulls  null_pct
digits        0       0.0
division      0       0.0
group         0       0.0
class         0       0.0
subclass      0       0.0
desc_en       0       0.0
desc_bm       0       0.0



## 3. Hidden missing values: `-1` codes

PriceCatcher's lookups use code `-1` for "Unknown". These rows are not null, so step 2 cannot catch them.

In [6]:
print("Lookup rows with code -1:")
print(lookup_item[lookup_item["item_code"] == -1].to_string())
print()
print(lookup_premise[lookup_premise["premise_code"] == -1].to_string())
print()

unknown = pd.DataFrame({
    "rows": [(prices["item_code"] == -1).sum(), (prices["premise_code"] == -1).sum()],
}, index=["item_code == -1", "premise_code == -1"])
unknown["pct_of_prices"] = (unknown["rows"] / len(prices) * 100).round(3)
unknown

Lookup rows with code -1:
   item_code item unit item_group item_category
0         -1  NaN  NaN        NaN           NaN

   premise_code          premise          address premise_type state district
0            -1  Unknown Premise  Unknown Address          NaN   NaN      NaN



,rows,pct_of_prices
item_code == -1,0,0.0
premise_code == -1,0,0.0


## 4. Price outliers

There are no prices of 0 or below, but some prices are clearly typing errors (for example RM 718 for 1 kg of carrots).

**Rule:** within each item, a price is an outlier if it is more than **5x** the item's median price, or less than **1/5** of it.

- The median is used because a single extreme typo barely moves it, unlike the mean.
- The goal is to catch impossible values, not unusual ones. The IQR rule was considered but flags about 3% of rows, including real price differences between premise types, which this project needs to compare.

In [7]:
print("Prices <= 0:", (prices["price"] <= 0).sum())
prices["price"].describe().round(2)

Prices <= 0: 0


count    22682970.00
mean           11.83
std            11.11
min             0.01
25%             5.00
50%             8.99
75%            14.50
max          7985.00
Name: price, dtype: float64

In [8]:
item_median = prices.groupby("item_code")["price"].transform("median")
ratio = prices["price"] / item_median

comparison = pd.DataFrame(
    {"rows_removed": [((ratio > k) | (ratio < 1 / k)).sum() for k in [3, 5, 10]]},
    index=["k = 3", "k = 5", "k = 10"],
)
comparison["pct_of_prices"] = (comparison["rows_removed"] / len(prices) * 100).round(3)
comparison

,rows_removed,pct_of_prices
k = 3,65259,0.288
k = 5,4252,0.019
k = 10,853,0.004


The most extreme prices flagged at k = 5, compared with their item's median:

In [9]:
K = 5
is_outlier = (ratio > K) | (ratio < 1 / K)

flagged = prices[is_outlier].assign(item_median=item_median[is_outlier], ratio=ratio[is_outlier])
flagged = flagged.merge(lookup_item[["item_code", "item", "unit"]], on="item_code")

columns = ["item", "unit", "price", "item_median", "ratio"]
print("Highest ratios:")
print(flagged.nlargest(5, "ratio")[columns].round(2).to_string(index=False))
print()
print("Lowest ratios:")
print(flagged.nsmallest(5, "ratio")[columns].round(3).to_string(index=False))

Highest ratios:
                                  item unit  price  item_median  ratio
BAWANG KECIL MERAH ROSE IMPORT (INDIA)  1kg 5390.0         6.90 781.16
                KACANG DAL (AUSTRALIA)  1kg 1250.0         5.00 250.00
                  KACANG SOYA (IMPORT)  1kg 1600.0         6.99 228.90
                         LIMAU KASTURI  1kg 1299.0         7.00 185.57
                KUBIS BULAT (TEMPATAN)  1kg  904.9         5.00 180.98

Lowest ratios:
                                                              item  unit  price  item_median  ratio
                        BERAS CAP JASMINE SUPER 5 SPECIAL (IMPORT) 10 kg   0.01        35.59  0.000
                                       MINYAK MASAK TULEN CAP ALIF  5 kg   0.01        30.90  0.000
                            DAGING KERBAU IMPORT (INDIA) * (CHUCK)   1kg   0.01        23.99  0.000
UDANG PUTIH BESAR (BERAT ANTARA 41 EKOR HINGGA 60 EKOR SEKILOGRAM)   1kg   0.02        42.00  0.000
                       SOTONG (ANTARA 11 H

## 5. Items with few prices

A per-item median is only reliable when an item has enough prices. Items with fewer than 30 prices across all the months loaded are listed here, so they can be avoided when choosing basket items.

In [10]:
item_counts = prices.groupby("item_code").size().rename("price_rows")
few = (
    item_counts[item_counts < 30]
    .reset_index()
    .merge(lookup_item[["item_code", "item", "unit"]], on="item_code")
    .sort_values("price_rows")
)
print(f"{len(few)} of {len(item_counts)} items have fewer than 30 prices")
print(few.to_string(index=False))

11 of 356 items have fewer than 30 prices
 item_code  price_rows                                             item   unit
        49           1 IKAN GELAMA (ANTARA 5 HINGGA 10 EKOR SEKILOGRAM)    1kg
        60           1              IKAN MERAH (b\t% 1 KILOGRAM SEEKOR)    1kg
       112           1                                   TERUNG PANJANG    1kg
       152           1                            KACANG HIJAU (IMPORT)    1kg
      1557           1                                      BAYAM MERAH    1kg
      1486           1                EPAL MERAH RED DELICIOUS (SAIZ M) 1 biji
      1922           1                                           SADERI    1kg
      1818           1                                     KACANG BOTOL    1kg
      1943           1                            KACANG MERAH (IMPORT)    1kg
      1399           4                     BAWANG PERAI (LEEK) TEMPATAN    1kg
      1419          23                         KACANG DAL BIASA (INDIA)    1kg


## 6. Units

Prices are not converted to a common unit (such as per kg). Every research question compares the **same item** over time, across premise types or across states, and each item has one fixed unit, so its prices are already comparable. Many units also cannot be converted to weight, such as `10 biji`, `1 liter` or `M64, L54, XL42`.

`unit` is kept as a label that describes what one price buys.

In [11]:
print("Each item_code has exactly one unit:", lookup_item["item_code"].is_unique)

used_items = lookup_item[lookup_item["item_code"].isin(prices["item_code"].unique())]
print(f"{len(used_items)} items appear in the prices, using {used_items['unit'].nunique()} different unit labels")
print()
print(used_items["unit"].value_counts().head(10).to_string())

Each item_code has exactly one unit: True
339 items appear in the prices, using 75 different unit labels

unit
1kg        155
10 kg       17
250 g       10
2 kg        10
400 g        8
5 kg         7
500 g        7
850g         6
1 liter      5
600 g        5


## 7. Stray spaces in text

`premise_type` has two versions of `Pasar Basah` (wet market), so wet markets would be split into two groups. The check below finds every text value with leading or trailing spaces.

In [12]:
print("Pasar Basah values before trimming:")
print(lookup_premise.loc[lookup_premise["premise_type"].str.contains("Basah", na=False), "premise_type"].map(repr).value_counts().to_string())
print()
print("After trimming:")
print(lookup_premise["premise_type"].str.strip().loc[lambda s: s.str.contains("Basah", na=False)].value_counts().to_string())

Pasar Basah values before trimming:
premise_type
'Pasar Basah '    338
'Pasar Basah'      22

After trimming:
premise_type
Pasar Basah    360


In [13]:
rows = []
for name, df in tables.items():
    for col in df.select_dtypes(include=["object", "string"]).columns:
        values = df[col].dropna().astype(str)
        rows.append({
            "table": name,
            "column": col,
            "values_with_spaces": (values != values.str.strip()).sum(),
            "distinct_before": values.nunique(),
            "distinct_after": values.str.strip().nunique(),
        })

spaces = pd.DataFrame(rows)
print(spaces[spaces["values_with_spaces"] > 0].to_string(index=False))

         table       column  values_with_spaces  distinct_before  distinct_after
lookup_premise premise_type                 338               13              12
       mcoicop      desc_en                   1              293             292


## 8. Lookup coverage

Every `item_code` and `premise_code` in the prices should exist in its lookup; otherwise the price cannot be named or placed in a state. This check was added after the database's foreign keys found prices for items missing from the item lookup.

In [14]:
missing_items = prices.loc[~prices["item_code"].isin(lookup_item["item_code"]), "item_code"]
missing_premises = prices.loc[~prices["premise_code"].isin(lookup_premise["premise_code"]), "premise_code"]

coverage = pd.DataFrame({
    "codes_missing": [missing_items.nunique(), missing_premises.nunique()],
    "price_rows": [len(missing_items), len(missing_premises)],
}, index=["item_code not in lookup_item", "premise_code not in lookup_premise"])
coverage["pct_of_prices"] = (coverage["price_rows"] / len(prices) * 100).round(2)
coverage

,codes_missing,price_rows,pct_of_prices
item_code not in lookup_item,17,337483,1.49
premise_code not in lookup_premise,0,0,0.00


In [15]:
missing_detail = (
    prices[prices["item_code"].isin(missing_items.unique())]
    .groupby("item_code")
    .agg(price_rows=("price", "size"), first_seen=("date", "min"), last_seen=("date", "max"),
         premises=("premise_code", "nunique"))
    .sort_values("price_rows", ascending=False)
)
print(missing_detail.to_string())

           price_rows  first_seen   last_seen  premises
item_code                                              
2057            39042  2025-09-08  2026-09-28       885
2094            37199  2025-09-08  2026-09-28       849
2053            34709  2025-09-08  2026-09-28       831
2059            34468  2025-09-08  2026-09-28       868
2095            31787  2025-09-08  2026-09-28       820
2089            31772  2025-09-08  2026-09-28       788
2090            27475  2025-09-08  2026-09-28       776
2091            23716  2025-09-08  2026-09-28       623
2023            21073  2025-09-08  2026-09-28       521
2092            20577  2025-09-08  2026-09-28       592
2096            18752  2025-09-08  2026-09-28       608
2093            16490  2025-09-08  2026-09-28       444
2048              252  2025-09-08  2026-09-28        10
2035              166  2025-09-08  2026-09-28        10
2070                3  2025-10-05  2025-11-03         2
2065                1  2025-11-03  2025-11-03   

## Summary

**What the data looks like**
- **22,682,970 PriceCatcher rows** (September 2025 to September 2026, 13 months) and **45,024 CPI rows**, with **no nulls** in either.
- The only lookup nulls are the `-1` "Unknown" rows, and **no price row uses a `-1` code**.
- `mcoicop` uses `-` instead of null for code levels that don't apply; only the 2-digit divisions are needed.
- PriceCatcher `date` is stored as text.
- **No prices of 0 or below**, but there are clear typing errors, such as RM 5,390 for 1 kg of shallots (median RM 6.90) and RM 0.01 for 10 kg of rice (median RM 35.59).
- Each item has **exactly one unit**; 339 items use 75 different unit labels, many of which cannot be converted to weight.
- `premise_type` has `Pasar Basah` twice because of a trailing space; one `mcoicop` description also has a stray space.
- **11 items have fewer than 30 prices** across the 13 months, mostly fresh produce with only 1 price.
- **17 item codes in the prices are missing from the item lookup**: 337,483 rows (1.49%). Most were first recorded on 8 September 2025 and are priced at up to 885 premises. They look like items DOSM has not described yet. Every premise code is covered.
- **Missing days are public holidays** (e.g. Malaysia Day, Merdeka, Chinese New Year, Hari Raya Aidilfitri), when prices are not collected. Every month has data across its whole span, so monthly medians are not biased, but festive-day prices themselves are not captured.

**Cleaning rules** (applied in the SQL clean layer in Phase 4; raw files stay unchanged)

1. **Months:** use September 2025 to September 2026. Thirteen months let price changes be compared year-on-year (September vs September), which cancels seasonal effects and matches the CPI analysis. October 2026 is not used because its file only covers 10–31 October.
2. **Unknown codes:** drop the `-1` rows from `lookup_item` and `lookup_premise`.
3. **Dates:** convert PriceCatcher `date` from text to a real date.
4. **Outliers:** remove prices more than 5x or less than 1/5 of their item's median. This flags 4,252 rows (0.019%) here; k = 3 would flag 65,259 rows, starting to include plausible prices, while k = 10 would keep obvious errors. The database removes 4,213, because 39 prices sit exactly at 1/5 of the median and floating-point division here makes them look slightly below it; the SQL keeps them, as the rule says.
5. **Text:** trim leading and trailing spaces in all text columns, which merges the two `Pasar Basah` values into one (360 premises).
6. **Units:** keep `unit` as a label and do not convert prices, because every analysis compares the same item.
7. **CPI categories:** use only the 2-digit `mcoicop` divisions `01`–`13`, read as text so the leading zero is kept.
8. **Lookup coverage:** leave prices for items missing from the item lookup out of the mart, because every research question reports results by item name. They stay in the raw and clean layers and return automatically once DOSM adds them to the lookup. In the database this removes 337,456 rows; the 27 others found here are also outliers removed by rule 4.

**Notes for later phases**
- For the Q4 basket, choose items with plenty of prices and clear units (such as `1kg` or `30 biji`), avoiding the low-count items listed in section 5.